In [2]:
import pandas as pd
import requests
import zipfile
import io

In [3]:
desc = ['Mkt-RF', 'SMB', 'HML']
datacomment = '(Simple monthly returns, 1926-2001)'

url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip"

# Hacemos la petición y leemos el ZIP en memoria
response = requests.get(url)
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    filename = z.namelist()[0] # Suele llamarse 'F-F_Research_Data_Factors.CSV'
    # Leemos el CSV saltando las primeras 3 líneas de texto introductorio
    df_raw = pd.read_csv(z.open(filename), skiprows=3)

# Limpieza del formato particular del CSV de Fama-French
df_raw.rename(columns={df_raw.columns[0]: 'Date'}, inplace=True)

# El CSV trae datos mensuales y luego anuales abajo del todo. 
df_raw['Date'] = pd.to_numeric(df_raw['Date'], errors='coerce')
df_raw = df_raw.dropna(subset=['Date'])
df_raw['Date'] = df_raw['Date'].astype(int).astype(str)

In [4]:
from config import RAW_DATA_DIR

# Filtramos para quedarnos solo con las filas que tienen una fecha mensual válida (YYYYMM) o años
df_raw_monthly = df_raw[df_raw['Date'].str.len() == 6] # Solo meses (ej: 192607)
df_raw_yearly = df_raw[df_raw['Date'].str.len() == 4] # Solo años (ej: 2021)

# ----- Datos mensuales -----
df_raw_monthly['Date'] = pd.to_datetime(df_raw_monthly['Date'], format='%Y%m')
df_raw_monthly.set_index('Date', inplace=True)
df_raw_monthly = df_raw_monthly.astype(float)

# ----- Datos anuales -----
df_raw_yearly['Date'] = pd.to_datetime(df_raw_yearly['Date'], format='%Y')
df_raw_yearly.set_index('Date', inplace=True)
df_raw_yearly = df_raw_yearly.astype(float)

# Guardamos por separado
df_raw_monthly.to_csv(RAW_DATA_DIR / "dataset_paper1_monthly.csv")
df_raw_yearly.to_csv(RAW_DATA_DIR / "dataset_paper1_yearly.csv")